# 05 - Rebuild analysis from the shared dataset

This notebook recomputes the existing Part B analysis using the shared, verified cleaned tables.
Run notebook 04 or `python scripts/build_clean_data.py --zip` first. It does not establish that a
report is correct, that observed group differences are causal, or that projections are validated.

The analysis window remains frozen at 28 August 2026. Product cohorts now require 730 complete days.
Search totals use the common source-selection policy, so outputs can differ from older drafts.
Review `docs/cleaning_contract.md` and the dataset manifest warnings before interpreting results.


## Setup

Set `WEAVVE_DATA` to the original sources and `WEAVVE_OUT` to the private output folder before starting
Jupyter. All current notebooks load the same manifest. Restart the kernel after rebuilding data or code.
Optional local ads snapshots extend Section 10. A Word comparison runs only when `WEAVVE_REPORT`
is explicitly set to a local file; this notebook does not discover or access online reports.


In [ ]:
import sys
sys.path.insert(0, "../src")          # our own helper code lives in the src folder

import json
import os
import runpy
from pathlib import Path

import pandas as pd
from IPython.display import display

from weavve import paths
from weavve.final import data, figures, report_numbers

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
CHECKS = Path(data.__file__).parent   # the check scripts sit next to data.py
FIGURES = paths.OUT / "final"         # Figures 5 to 7 are saved here
SNAPSHOT = "2026-10-31"               # the ads monitor snapshot that Section 10 of the report uses
print("Data folder:", paths.DATA)
print("Figures folder:", FIGURES)
shared = data.dataset()
print("Shared dataset ID:", shared.manifest["dataset_id"])
assert shared.manifest["transaction_cutoff"] == "2026-08-28", "Review the frozen analysis windows for this new data cutoff."


## 1. Run the existing analysis formulas

The four scripts use canonical orders and support tables through `weavve.final.data`.
Net and discount amounts already include the line-level correction; the scripts do not repeat it.
`check_recs` rebuilds customer/product summaries, `check_inputs` the recommendation inputs,
`check_team` the loyalty comparisons, and `check_rec1` the channel/profile cohorts.

Source hashes establish provenance, not agreement with a report. Profile creation remains a proxy
for signup; club groups and list labels are analysis assumptions. Neither establishes consent,
complete membership history, or a treatment effect.


In [ ]:
def run(check, name="res"):
    """Runs one check script and returns the results it builds."""
    return runpy.run_path(str(CHECKS / f"{check}.py"))[name]


R = {
    "recs": run("check_recs", "results"),
    "inputs": run("check_inputs"),
    "team": run("check_team"),
    "rec1": run("check_rec1"),
    "funnel": data.funnel_monthly(),
}
R = json.loads(json.dumps(R, default=str))   # text keys and plain values, as the checks first saved them
pd.DataFrame(data.SOURCES).T
print("Search split coverage (only listed queries with observed numeric clicks):")
display(pd.DataFrame(R["recs"]["search_clicks_janjul"]).T)


## 2. Recomputed excerpts by section

These are computed values formatted in the existing report layout. Wording inherited from the analysis
is not independently verified narrative. Margin, acquisition, cohort and uplift assumptions still need
review. Without the optional local ads monitor snapshots, Section 10 is explicitly omitted.


In [ ]:
from hashlib import sha256

SNAPSHOT_SOURCES = {}

def snapshot(kind):
    """The ads monitor snapshot of this kind, or None if it is not in the data folder."""
    snapshot_root = Path(os.environ.get("WEAVVE_ADS", str(paths.DATA)))
    found = sorted(snapshot_root.rglob(f"ads_monitor_{kind}_{SNAPSHOT}.json"))
    if len(found) > 1:
        raise ValueError(f"Ambiguous optional snapshot: {kind}")
    if not found:
        return None
    payload = found[0].read_bytes()
    SNAPSHOT_SOURCES[kind] = {"file": found[0].name, "sha256": sha256(payload).hexdigest()}
    return json.loads(payload)


ads, log, plan = snapshot("api_data"), snapshot("api_log"), snapshot("plan")
q = report_numbers.quotes(R, ads, log, plan)
print(f"The checks rebuilt {len(q)} numbers.")
if not (ads and log and plan):
    print("The ads monitor snapshots are not in the data folder, so Section 10 is left out.")
# This separate manifest records optional evidence that is not part of the core cleaned tables.
FIGURES.mkdir(parents=True, exist_ok=True)
analysis_manifest = {
    "dataset_id": shared.manifest["dataset_id"],
    "analysis_cutoff": "2026-08-28",
    "analysis_code": {p.name: sha256(p.read_bytes()).hexdigest() for p in sorted(CHECKS.glob("*.py"))},
    "optional_ads_sources": SNAPSHOT_SOURCES,
    "optional_ads_complete": bool(ads and log and plan),
    "pandas_version": pd.__version__,
}
(FIGURES / "analysis_manifest.json").write_text(json.dumps(analysis_manifest, indent=2), encoding="utf-8")
print("Optional ads evidence:", SNAPSHOT_SOURCES or "not supplied")


In [ ]:
SECTIONS = {"6": "6. What We Found", "7": "7. Recommendation 1: Win First Website Orders",
            "8": "8. Recommendation 2: Snooze Club Worth Joining", "9": "9. Order of Actions and Shared Measures",
            "10": "10. Paid-Search Campaign", "11": "11. Assumptions and Limits"}
TABLE_SECTION = {10: "6", 11: "6", 12: "6", 13: "7", 14: "7", 15: "7", 16: "7", 17: "8", 18: "8", 19: "8", 20: "8",
                 21: "8", 22: "9", 25: "10", 26: "10"}
FIGURE_SECTION = {5: "6", 6: "7"}


def section(where):
    """The report section that holds a number, from its section number or its table."""
    first, second = where.split(" ")[:2]
    if first in ("Table", "Figure"):
        return (TABLE_SECTION if first == "Table" else FIGURE_SECTION)[int(second)]
    return first.split(".")[0]


rebuilt = pd.DataFrame({"Where": list(q), "As printed": list(q.values())})
rebuilt["Section"] = rebuilt["Where"].map(section)
for key, title in SECTIONS.items():
    part = rebuilt[rebuilt["Section"] == key]
    if len(part):
        print(title)
        display(part[["Where", "As printed"]].reset_index(drop=True))

## 3. Recomputed figures

Figures 5 and 6 use the shared results. Figure 7 requires the optional ads snapshots. Charts are saved
only in the private output folder. They are descriptive evidence, not proof of an intervention effect.


In [ ]:
figures.figure5(R["recs"]["online_store"]["lens5"], FIGURES)
figures.figure6(R["rec1"]["new_signups_by_quarter"], FIGURES)
if ads and plan:
    figures.figure7(ads, plan["baseline"]["ctr"], FIGURES)

## 4. Optional local excerpt comparison

Set `WEAVVE_REPORT` to an explicitly selected local Word file to compare calculated excerpts.
Table 11 checks associate values with their product row and metric column. Other checks match text
fragments within the specified table or document; they do not establish correct context, definitions,
complete coverage, citations, or causal interpretation. Missing excerpts fail visibly.


In [ ]:
report_file = Path(os.environ["WEAVVE_REPORT"]) if os.environ.get("WEAVVE_REPORT") else None
if report_file is None:
    print("Local report comparison skipped: WEAVVE_REPORT is not set. No report has been verified.")
else:
    missing = report_numbers.compare(q, report_numbers.report_text(report_file))
    print(f"{len(q) - len(missing)} of {len(q)} calculated excerpt checks matched {report_file.name}.")
    if missing:
        display(pd.DataFrame({"Where": list(missing), "Rebuilt": list(missing.values())}))
        raise AssertionError(f"{len(missing)} report excerpts do not match the recomputed analysis.")
